In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
# sys.path.append('/image-differential-annotator/')
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator')

from annotator.annotation import runAnnotation, jumpStart
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProbImg

from tqdm import tqdm
import numpy as np
import pandas as pd
import pickle
import scanpy as sc

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

classifierPaths = 'classifiers/'
if not os.path.exists(classifierPaths):
    os.makedirs(classifierPaths)

In [2]:
set1 = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf', '#999999']
set2 = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
set3 = ['#8dd3c7', '#ffffb3', '#bebada', '#fb8072', '#80b1d3', '#fdb462', '#b3de69', '#fccde5', '#d9d9d9', '#bc80bd', '#ccebc5', '#ffed6f']
Set123 = ListedColormap(set1 + set2 + set3)
Set123

In [21]:
outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-STQ-post-xenium-32-final/'
samples  = ['JDC-WP-012-ae', 'JDC-WP-012-w', 'JDC-WP-002-v', 'JDC-WP-002-r']

df_ids = pd.read_csv('/sdata/activities/kappsen-tmc/visium/analysis/downstream/Pancreas Xenium identifiers and paths.csv', index_col=0)
IDsToSTQnames = df_ids['HE-slide-identifier'].to_dict()

xeniumBundlePaths = df_ids['Xenium-CODEX-outs'].dropna().str.replace('/xe/', '/xe/spaceranger-custom-seg-outs/').to_dict()
xeniumMatrixPaths = df_ids['HE-matrix'].to_dict()

In [22]:
cell_barcodes = {s: pd.read_csv(xeniumBundlePaths[s] + '/analysis/clustering/gene_expression_kmeans_5_clusters/clusters.csv')['Barcode'].astype(str) for s in samples}
cell_barcodes_inv = {s: {v: k for k, v in cell_barcodes[s].items()} for s in samples}

In [23]:
annotations = {s: (pd.read_csv(xeniumBundlePaths[s] + '/analysis/clustering/gene_expression_kmeans_5_clusters/clusters.csv')['Cluster'].astype(str) + '').astype('category') for s in samples}
# annotations = {s: pd.read_csv(xeniumBundlePaths[s] + '/analysis/clustering/gene_expression_graphclust/clusters.csv')['Cluster'].astype(str).astype('category') for s in samples}

In [24]:
p = '/sdata/activities/kappsen-tmc/visium/analysis/downstream/pancreas-Xenium-pre-processed-32'
annotations = {}
for sample in samples:
    se = sc.read_h5ad(p + f'/xenium-{sample}-v2-processed.h5ad').obs['Celltype fine']
    se.index = se.index.str.split('.').str[0]
    se.index = [cell_barcodes_inv[sample][i] if i in cell_barcodes_inv[sample] else i for i in se.index]
    annotations[sample] = se

In [25]:
# If L is None, then the annotator will do lazy loading of patches with Zarr, else load entire images at requested level L
L = None
ts = 56 # Center-to-center distance between tiles (not size of a tile!)
mpp = 0.25 # Image pixel size
N = 8 # patch size, e.g., 8 by 8 tiles
fname='img.data.ctranspath-1.h5ad' # 'img.data.ctranspath-1.h5ad' or 'features/false-1-ctranspath_features.tsv.gz'

# Load the STQ data for each sample
ads = {}
imgs = {}
for sample in tqdm(samples):
    ads[sample], imgs[sample] = loadAd(f'{outsSTQpath}{IDsToSTQnames[sample]}/', fname=fname, L=L)

# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=N, spacing=ts/mpp, sample_id=sample) for sample in tqdm(samples)], axis=0) # N=8

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [ ]:
startParams = {}
jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=6, nrows=5, ads=ads, L=1 if L is None else L, sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3)

In [37]:
clf, plog, bp = {}, [], {}
# try: bp.update({startParams['selected_patch']: 'positive'})
# except: pass
runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1,
            figsize=(5, 5), augFunc=PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'], # (5, 5)
            seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp,
            xeniumBundlePaths=xeniumBundlePaths, xeniumMatrixPaths=xeniumMatrixPaths, numColumns=1,
            transcriptsAlpha=0.05, transcriptsColor='lime', transcriptsSize=2., loadTranscripts=False,
            selectedGenes=None, minCount=1, loadCells=True, loadCellBoundaries=True,
            showAnnotations=False, annotations=annotations, annotationsPalette=Set123)
# ['PPY', 'GCG', 'SST', 'INS', 'TTR']
# 'TM4SF4', 'CHGA', 'CHGB',

In [ ]:
# To save the classifier
if False:
    with open(f'{classifierPaths}/clf-PPY-012.pkl', 'wb') as tempfile:
        pickle.dump(clf['clf'], tempfile)

# To load the classifier back
if False:
    clf = {}
    with open(f'{classifierPaths}/clf-somename.pkl', 'rb') as tempfile:
        clf['clf'] = pickle.load(tempfile)

In [28]:
# To run inference on the entire slides
for i in range(len(samples)):
    infSample  = samples[i]
    x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
    showProbImg(x, y, p, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)

In [ ]:
# To run inference on the entire slides and save the images, and then display them in a grid
for i in range(len(samples)):
    infSample  = samples[i]
    x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
    showProbImg(x, y, p, f=1, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample,
                saveName=f'{infSample}.png', dpi=150)

nc = 4
nr = int(np.ceil(len(samples) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(nc*3, nr*3))
axs = axs.flatten()
for i in range(nr*nc):
    ax = axs[i]
    if i < len(samples):
        img = plt.imread(f'{samples[i]}.png')
        ax.imshow(img)
    ax.axis('off')
fig.tight_layout()
plt.show()